<a href="https://colab.research.google.com/github/udeme-sml/AIassistant/blob/main/S%C3%A9curit%C3%A9_informatique_Devoir_3_A2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Devoir 3 — Cryptanalyse

**IFT3275 / IFT6271 — Sécurité informatique — Automne 2026**

Ce carnet accompagne l'énoncé du devoir 3. L'énoncé dit quoi faire. Le carnet contient le détail technique : l'encodage du texte, le code Python des systèmes à briser et les données de chaque question.

Le devoir comporte trois questions. Dans chacune, un système de chiffrement est mal utilisé, et vous devez retrouver le message sans connaître la clé.

| Question | Système à briser | Chapitre du cours |
|---|---|---|
| 1 | RSA naïf, sans remplissage | La cryptographie à clé publique : confidentialité |
| 2 | Chiffre par substitution sur 256 symboles | Une histoire des codes secrets |
| 3 | Masque jetable employé deux fois | La cryptographie à clé secrète : le masque jetable |

Ce travail n'est pas à remettre. Il sera évalué aux examens.

## Comment utiliser ce carnet

1. Faites-en une copie : menu **Fichier**, puis **Enregistrer une copie dans Drive**. Vous travaillez ensuite dans votre copie.
2. Exécutez les cellules de code dans l'ordre, du haut vers le bas. Chaque section utilise les fonctions définies dans les sections qui précèdent.
3. Écrivez vos solutions dans les cellules « Votre solution », à la fin de chaque question.

Ce carnet est prévu pour Colab. Si vous l'exécutez sur votre propre ordinateur, installez d'abord le module `requests`, avec la commande `pip install requests`.

# Encodage du texte

Les questions 1 et 3 chiffrent du texte. Il faut donc une convention pour transformer un texte en bits.

**Un caractère, un octet.** En Python, `ord(c)` donne le numéro du caractère `c`, et `chr(n)` donne le caractère de numéro `n`. Les messages des questions 1 et 3 n'emploient que des caractères dont le numéro est inférieur à 256 : les lettres, les chiffres, la ponctuation et les lettres accentuées du français. Chaque caractère est donc représenté par son numéro, écrit sur 8 bits.

```
ord("A") = 65  = 01000001
ord("é") = 233 = 11101001
```

Attention : ce n'est pas l'encodage UTF-8, dans lequel « é » occupe deux octets. Ici, chaque caractère occupe exactement un octet.

**Un message, une chaîne de bits.** Un message est encodé en mettant bout à bout les 8 bits de chacun de ses caractères.

**Un message, un entier.** Pour la question 1, cette chaîne de bits est lue comme un entier écrit en base 2.

```
"Alain Tapp"
= [A, l, a, i, n, espace, T, a, p, p]
= [65, 108, 97, 105, 110, 32, 84, 97, 112, 112]
= 01000001 01101100 01100001 01101001 01101110 00100000 01010100 01100001 01110000 01110000
= 308953089009066937774192
```

In [ ]:
def texte_vers_octets(M):
    """Texte -> liste d'entiers de 0 à 255, un par caractère."""
    octets = [ord(c) for c in M]
    for c, n in zip(M, octets):
        if n >= 256:
            raise ValueError(f"Le caractère {c!r} (numéro {n}) ne tient pas sur 8 bits.")
    return octets

def octets_vers_texte(octets):
    """Liste d'entiers de 0 à 255 -> texte."""
    return "".join(chr(n) for n in octets)

def texte_vers_bits(M):
    """Texte -> chaîne de bits, 8 bits par caractère."""
    return "".join(f"{n:08b}" for n in texte_vers_octets(M))

def texte_vers_entier(M):
    """Texte -> entier : la chaîne de bits lue en base 2."""
    n = 0
    for octet in texte_vers_octets(M):
        n = n * 256 + octet
    return n

def entier_vers_texte(n):
    """Entier -> texte : l'opération inverse de texte_vers_entier."""
    octets = []
    while n > 0:
        octets.append(n % 256)
        n = n // 256
    return octets_vers_texte(reversed(octets))

In [ ]:
# Exemples
for c in ["A", "é", "!"]:
    print(c, "=", ord(c), "=", f"{ord(c):08b}")

M = "Alain Tapp"
print()
print(M)
print(texte_vers_octets(M))
print(texte_vers_bits(M))
print(texte_vers_entier(M))
print(entier_vers_texte(texte_vers_entier(M)))

# Question 1 — RSA naïf

**Rappel du cours.** La clé publique est $(N, e)$ et la clé privée est $d$. Le message $M$, un entier plus petit que $N$, devient le cryptogramme

$$C = M^e \bmod N.$$

On parle de *RSA naïf* quand on chiffre directement l'entier $M$, sans schéma de remplissage. Le chapitre « La cryptographie à clé publique : confidentialité » montre, dans la section « Sécurité de RSA », que cette façon de faire a des failles. Vous allez en exploiter deux.

Dans les deux sous-questions, le message est un nom, encodé en entier comme à la section « Encodage du texte ». Vous connaissez la clé publique et le cryptogramme. Vous ne connaissez pas la clé privée, et les modules ont 2048 bits : il n'est pas question de les factoriser.

**Question 1.1.** Le nom d'un personnage célèbre a été chiffré avec l'exposant $e = 3$. De qui s'agit-il ?

**Question 1.2.** Le nom d'un auteur célèbre a été chiffré avec l'exposant $e = 173$. De qui s'agit-il ? Cette question ressemble à la précédente, mais elle est nettement plus difficile : la méthode qui règle la question 1.1 ne fonctionne plus.

**Pour vérifier une réponse**, chiffrez-la avec la clé publique et comparez le résultat au cryptogramme.

## Outils

Ce sont les fonctions Python du cours. Python offre déjà ces calculs : `math.gcd(a, b)`, `pow(a, -1, m)` et `pow(a, x, m)`.

In [ ]:
def pgcd(a, b):
    """Plus grand commun diviseur, par l'algorithme d'Euclide."""
    while b > 0:
        a, b = b, a % b
    return a

def inverse(a, m):
    """Inverse multiplicatif de a modulo m. Il faut que pgcd(a, m) = 1."""
    a, b, c, d = m, a, 1, 0
    while b != 1:
        a, b, c, d = b, a % b, (d - c * (a // b)) % m, c
    return c

def puissance(a, x, m):
    """a à la puissance x, modulo m, par exponentiation rapide."""
    r = 1
    while x > 0:
        if x % 2 == 1:
            r = r * a % m
        a = a * a % m
        x = x // 2
    return r

def chiffrer_rsa(M, N, e):
    """RSA naïf : le texte M est encodé en entier, puis élevé à la puissance e modulo N."""
    return puissance(texte_vers_entier(M), e, N)

# Exemples du cours
print(pgcd(42, 30), inverse(5, 13), puissance(5, 21, 17))

## Données de la question 1.1

In [ ]:
# Clé publique de la question 1.1
N_11 = 24796598206773644387290864970319298407875430175358028089373755190329743547643105632323697873288396842336229982516193304578132256646806693826299717921488879485455157467791099242789483045820875424989344744912779939884684382392037419089009118600558468205265378409430623633998631849856297872173564921672409261932422726067513422489820324868390089171787240950667391254303346556831725691626627173890296329211063856609941487970397803339085751962593422759653356511967326452037956638361179574909701588267890275453150976932007792207442639887226852068477172610384714454445210796442010275423741270999465848383245143149119066199437
e_11 = 3

# Cryptogramme de la question 1.1
C_11 = 8288860847968368628983990533299738740620513824609403286271350376705543689123592436125

## Données de la question 1.2

In [ ]:
# Clé publique de la question 1.2
N_12 = 24563014793610276175361007073988448647266048166246520585044339683298733494780575915828687199013665384274992258222545147447796863786718304505399256875615152246186551782881049669998755330887370424535850912964947622668409569598625823454576452509377997070625671840168208415906711904985468731091818048003863024028458054517237290194775393082078589593177429848690888095797166161876533316748931138889077781567991599980425264155575047704072274363023227987105537946272179742246701642633867428678995528674247259368734735137946363352338321945517019707733987631476836241849541750234520381684380823351873060318648585269388842859601
e_12 = 173

# Cryptogramme de la question 1.2
C_12 = 6960075451251499298706105917988946648330001113575934856722843625662348051110354912983720760954795044610498984046729240262633058416201547729237680304800740407551214364449135145902878543478964722960570695043130369753247104583410330967843095462752508221983947684572357862399694369687543036669829163559893740519464903667336096338872094852408060394351082049773672323282567281628436972465728221465956929810620336699229934748161493789075955904304702065469871198259827093215009218880192183045778231934410011601564548539337026896461728761673781917225773212304690949070527635954087825023403029584502439416680025304632171938644

## Votre solution à la question 1

In [ ]:
# Question 1.1 : votre code ici


# Question 1.2 : votre code ici

# Question 2 — Substitution par symboles

**Rappel du cours.** Dans un chiffre par substitution, chaque lettre du message est remplacée par une autre, toujours la même. Le chapitre « Une histoire des codes secrets » montre comment l'analyse des fréquences brise ce chiffre : la lettre la plus fréquente du cryptogramme cache la lettre la plus fréquente de la langue.

Le système de cette question est une substitution plus riche. Son alphabet compte 256 symboles, et certains symboles sont des paires de caractères. Est-ce que cela suffit à cacher les fréquences de la langue ? À vous de montrer que non.

**Le système.** Selon le principe de Kerckhoffs, tout est public, sauf la clé.

* **Symboles.** Un *caractère* est un caractère du texte. Un *bicaractère* est une paire de caractères. L'ensemble des symboles réunit tous les caractères d'un corpus de textes français et les bicaractères les plus fréquents de ce corpus, pour un total de 256 symboles. Cette liste est fixe et publique : elle fait partie du système.
* **Clé.** La clé associe à chacun des 256 symboles un octet distinct. Elle est tirée au hasard, uniformément, parmi les $256!$ possibilités, soit environ $2^{1684}$ clés. Essayer toutes les clés est donc exclu.
* **Chiffrement.** Le message est découpé en symboles, de gauche à droite. Si les deux prochains caractères forment un bicaractère de la liste, on prend la paire. Sinon, on prend le caractère seul. Chaque symbole est ensuite remplacé par son octet. Le cryptogramme est la chaîne de bits obtenue, 8 bits par symbole.

**Votre tâche.** Écrivez une fonction `dechiffrer(C)` qui reçoit un cryptogramme, c'est-à-dire une chaîne de bits, et qui retourne le texte clair, sans connaître la clé. Tous les messages sont des textes en français. Un banc d'essai, à la fin de cette section, mesure la réussite de votre fonction sur des messages tirés d'un livre qui ne fait pas partie du corpus.

## Le corpus

Le corpus réunit deux livres du [projet Gutenberg](https://www.gutenberg.org/) : le *Discours de la méthode* de Descartes et *Candide* de Voltaire. Vous pouvez y ajouter d'autres textes français pour vos propres statistiques.

In [ ]:
import requests
from collections import Counter

def telecharger(url):
    """Télécharge un texte et normalise les fins de ligne."""
    reponse = requests.get(url)
    reponse.raise_for_status()
    reponse.encoding = "utf-8"
    return reponse.text.replace("\r\n", "\n")

URL_DESCARTES = "https://www.gutenberg.org/ebooks/13846.txt.utf-8"   # Discours de la méthode
URL_VOLTAIRE  = "https://www.gutenberg.org/ebooks/4650.txt.utf-8"    # Candide

corpus = telecharger(URL_DESCARTES) + telecharger(URL_VOLTAIRE)
print("Taille du corpus en caractères :", len(corpus))
print()
print(corpus[10000:10500])

## Les 256 symboles

Cette cellule construit la liste des symboles : d'abord tous les caractères du corpus, puis les bicaractères les plus fréquents, jusqu'à 256 symboles. Les bicaractères sont comptés en découpant le corpus en paires successives. D'autres choix seraient possibles, mais pour ce devoir la liste est celle-ci.

In [ ]:
caracteres = sorted(set(corpus))
nb_bicaracteres = 256 - len(caracteres)

paires = [corpus[i:i + 2] for i in range(0, len(corpus) - 1, 2)]
bicaracteres = [paire for paire, _ in Counter(paires).most_common(nb_bicaracteres)]

symboles = caracteres + bicaracteres

print("Nombre de caractères   :", len(caracteres))
print("Nombre de bicaractères :", len(bicaracteres))
print("Nombre de symboles     :", len(symboles))
print()
print("Caractères   :", caracteres)
print()
print("Bicaractères :", bicaracteres)

## La clé

Une clé est un dictionnaire Python : à chaque symbole, il associe une chaîne de 8 bits, différente pour chaque symbole. La clé est tirée avec le hasard du système d'exploitation, qui n'a pas de graine à deviner : inutile de chercher une faiblesse du côté du générateur.

In [ ]:
import random
hasard = random.SystemRandom()

def generer_cle():
    """Tire une clé au hasard : un octet distinct pour chacun des 256 symboles."""
    octets = hasard.sample(range(256), 256)
    return {symbole: f"{octet:08b}" for symbole, octet in zip(symboles, octets)}

K = generer_cle()
print("Taille de la clé :", len(K), "symboles")
print(K)

## Chiffrer, et déchiffrer avec la clé

In [ ]:
ensemble_symboles = set(symboles)

def decouper(M):
    """Découpe le message en symboles, de gauche à droite : la paire si elle est dans la liste, sinon le caractère."""
    resultat = []
    i = 0
    while i < len(M):
        if i + 1 < len(M) and M[i:i + 2] in ensemble_symboles:
            resultat.append(M[i:i + 2])
            i += 2
        elif M[i] in ensemble_symboles:
            resultat.append(M[i])
            i += 1
        else:
            raise ValueError(f"Le caractère {M[i]!r} ne fait pas partie des symboles du système.")
    return resultat

def chiffrer(M, K):
    """Chiffre le message M avec la clé K. Le cryptogramme est une chaîne de bits."""
    return "".join(K[symbole] for symbole in decouper(M))

def dechiffrer_avec_cle(C, K):
    """Déchiffre le cryptogramme C quand on connaît la clé K."""
    K_inverse = {bits: symbole for symbole, bits in K.items()}
    return "".join(K_inverse[C[i:i + 8]] for i in range(0, len(C), 8))

In [ ]:
# Exemple
K = generer_cle()
M = corpus[10000:10100]
C = chiffrer(M, K)

print("M =", repr(M))
print("\nLongueur du message en caractères =", len(M))
print("\nDécoupage en symboles =", decouper(M))
print("\nNombre de symboles =", len(decouper(M)))
print("\nC =", C)
print("\nLongueur du cryptogramme en bits =", len(C))
print("\nLongueur du cryptogramme en octets =", len(C) // 8)
print("\nDéchiffrement avec la clé correct :", dechiffrer_avec_cle(C, K) == M)

## Votre solution à la question 2

Complétez la fonction `dechiffrer`. Elle ne reçoit que le cryptogramme. Elle peut utiliser tout ce qui est public : la liste des symboles, le corpus et tous les textes français que vous voulez.

In [ ]:
def dechiffrer(C):
    """Reçoit un cryptogramme (chaîne de bits) et retourne le texte clair, sans la clé."""
    M = ""
    # Votre code ici
    return M

## Banc d'essai

Le banc d'essai tire une clé et un extrait d'un livre qui ne fait pas partie du corpus, *Les Trois Mousquetaires* d'Alexandre Dumas. Il chiffre l'extrait, appelle votre fonction `dechiffrer` et compte les caractères retrouvés à la bonne position. Tant que la fonction n'est pas écrite, il affiche 0 %.

In [ ]:
URL_ESSAI = "https://www.gutenberg.org/ebooks/13951.txt.utf-8"   # Les Trois Mousquetaires

# On retire du texte d'essai les caractères absents du système.
ensemble_caracteres = set(caracteres)
texte_essai = "".join(c for c in telecharger(URL_ESSAI) if c in ensemble_caracteres)

def evaluer(dechiffrer, longueur=5000, essais=3):
    """Mesure la proportion de caractères retrouvés, sur plusieurs extraits et plusieurs clés."""
    total = 0
    for i in range(essais):
        debut = hasard.randrange(len(texte_essai) - longueur)
        M = texte_essai[debut:debut + longueur]
        C = chiffrer(M, generer_cle())
        D = dechiffrer(C)
        bons = sum(1 for a, b in zip(M, D) if a == b)
        total += bons / len(M)
        print(f"Essai {i + 1} : {bons} caractères corrects sur {len(M)}, soit {100 * bons / len(M):.1f} %")
    print(f"Moyenne : {100 * total / essais:.1f} %")

evaluer(dechiffrer)

# Question 3 — Masque jetable réutilisé

**Rappel du cours.** Avec le masque jetable, la clé $K$ est une suite de bits uniforme, aussi longue que le message, et le chiffrement est un ou exclusif bit à bit :

$$E(K, M) = M \oplus K.$$

Le chapitre « La cryptographie à clé secrète : le masque jetable » démontre que ce chiffre offre une sécurité parfaite, à une condition : la clé ne sert qu'une seule fois.

Ici, cette condition est violée. Deux messages $M_1$ et $M_2$ ont été chiffrés avec le **même** masque $K$ :

$$C_1 = M_1 \oplus K, \qquad C_2 = M_2 \oplus K.$$

Les deux messages sont des phrases en français de 150 caractères chacune, encodées à raison d'un octet par caractère, comme à la section « Encodage du texte ». Vous connaissez $C_1$ et $C_2$. Vous ne connaissez pas $K$.

**Votre tâche.** Retrouvez les deux messages, et expliquez votre méthode.

Les cryptogrammes sont donnés en hexadécimal : deux chiffres hexadécimaux par octet.

## Outils

In [ ]:
import hashlib

def hex_vers_octets(h):
    """Chaîne hexadécimale -> liste d'entiers de 0 à 255."""
    return list(bytes.fromhex(h))

def xor(a, b):
    """Ou exclusif de deux listes d'octets, élément par élément."""
    return [x ^ y for x, y in zip(a, b)]

def empreinte(M):
    """Empreinte SHA-256 d'un texte, encodé à raison d'un octet par caractère."""
    return hashlib.sha256(bytes(texte_vers_octets(M))).hexdigest()

# Exemple : chiffrer et déchiffrer avec un masque jetable
M = texte_vers_octets("Bonjour")
K = [hasard.randrange(256) for _ in M]
C = xor(M, K)
print(C)
print(octets_vers_texte(xor(C, K)))

## Données de la question 3

In [ ]:
C1 = hex_vers_octets("fdab1afca8da10e95292f90a6801ecd3318561e0fa784e6da410653a4f48e3fbcf19eef6ae20fd61b875a0f7557f70a579ab5d14984393b617efc7b47dc5cebf3c1ad7cb5fd9a796cdc01d4eb742c488b75ff5b2d5eea77dad8c408662392d2a763d57ff7797730c5a91914b58092cdc02f67903cf77ad6aff9b70e56d22e39f6cf19c122e5b4e6cd361329ea8e749e11a98486170d5")

C2 = hex_vers_octets("f2b14fede94044e458dce64ba1167dc930dc32a1f97fd43ee819207e5054adebc44ce3fce33ef77daf75a0e65e3a32a765af4912cbd999e314aecbbb7b45c5b3780e82c81adcbb1acd850748b346d2c8f24dace6c5f5ba7daa9650d52873a02a336d5de421926810138f810c511267920bb73310d374b528be8a2fb07370ef9f687c8004385b5362d3612e9fb5e749b35b8b407d71d5")

print(len(C1), len(C2))

## Pour vérifier vos réponses

Quand vous pensez avoir retrouvé un message au complet, calculez son empreinte avec la fonction `empreinte` et comparez-la à celle-ci. Une seule différence, même une majuscule ou un espace, change toute l'empreinte.

In [ ]:
EMPREINTE_M1 = "aa6f82289843568817881f82d4190252cd255a77e74702c6508cd30d1f93c4a2"
EMPREINTE_M2 = "c63405aaab3ba54a47719a2c40698e7052d2e78dbca613e4e05573c8b34d30b1"

# Exemple : print(empreinte(votre_M1) == EMPREINTE_M1)

## Votre solution à la question 3

In [ ]:
# Votre code ici

# Ressources permises

Vous pouvez utiliser toutes les ressources que vous jugez utiles, y compris l'intelligence artificielle. Ces ressources doivent vous aider à comprendre le matériel, et non à remplacer votre réflexion. Les examens se déroulent à livre fermé : vous devrez être capable de raisonner et d'expliquer ces attaques sans assistance.